# Analiza rynku nieruchomości — transakcje

Notebook czyści i analizuje dane transakcyjne z rynku nieruchomości oraz generuje interaktywny raport HTML.

**Wymagane pliki wejściowe** (w katalogu `data/`):
- `transakcje.csv` — surowe dane transakcyjne
- `tr2.csv` — dodatkowe atrybuty transakcji (sposób użytkowania, przeznaczenie w MPZP)

**Pliki wyjściowe** (zapisywane do katalogu `output/`):
- `dane_obrobione.csv` — oczyszczony i wzbogacony zbiór danych
- `dane_wmpzp_long.csv` — dane w formacie długim wg przeznaczenia w MPZP
- `histogramy_cena_brutto.png` — histogramy cen wg kategorii
- `raport_interaktywny.html` — interaktywny raport HTML z wynikami analiz


## 1. Importy i konfiguracja ścieżek

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)


## 2. Wczytanie i czyszczenie danych transakcyjnych

In [ ]:
data = pd.read_csv(DATA_DIR / "transakcje.csv")


In [ ]:
data["data_transakcji"] = pd.to_datetime(data["data_transakcji"], errors="coerce")

nieudane = data["data_transakcji"].isna().sum()
print(f"Nie sparsowano daty: {nieudane} wierszy ({nieudane/len(data):.2%})")

data["rok"] = data["data_transakcji"].dt.year
data = data[(data["rok"] >= 2015) & (data["rok"] <= 2026)].copy()

In [ ]:
df_clean = data.drop_duplicates()

In [ ]:
df_clean = df_clean[(df_clean['powierzchnia'] > 0) & (df_clean['cena_brutto'] > 0)]

In [ ]:
# Odrzucenie wartości odstających metodą IQR, osobno dla każdej kategorii nieruchomości
def iqr_granice(seria, k=1.5):
    Q1, Q3 = seria.quantile(0.25), seria.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - k*IQR, Q3 + k*IQR

df_clean['cena_ok'] = True
df_clean['powierzchnia_ok'] = True

for kat in df_clean['kategoria'].unique():
    mask_kat = df_clean['kategoria'] == kat

    # --- cena: dolna granica z IQR, górna granica jako 97. percentyl ---
    ceny_w_kategorii = df_clean.loc[mask_kat, 'cena_brutto']
    
    # Wyciągamy tylko dolną granicę z IQR (górną granicę wyznaczamy inaczej, patrz niżej)
    dolna, _ = iqr_granice(ceny_w_kategorii)
    
    # Górna granica to odcięcie najdroższych transakcji (97. percentyl)
    gorna = ceny_w_kategorii.quantile(0.97)
    
    poza_cena = mask_kat & ((df_clean['cena_brutto'] < dolna) | (df_clean['cena_brutto'] > gorna))
    df_clean.loc[poza_cena, 'cena_ok'] = False

    # --- powierzchnia: standardowe granice IQR ---
    valid = mask_kat & (df_clean['powierzchnia'] > 0)
    dolna_pow, gorna_pow = iqr_granice(df_clean.loc[valid, 'powierzchnia'])
    poza_pow = valid & ((df_clean['powierzchnia'] < dolna_pow) | (df_clean['powierzchnia'] > gorna_pow))
    df_clean.loc[poza_pow, 'powierzchnia_ok'] = False

# Końcowe przefiltrowanie
df_final = df_clean[df_clean['cena_ok'] & df_clean['powierzchnia_ok']].copy()

In [ ]:
len(df_final)


In [ ]:
df_final.describe()


## 3. Zapis oczyszczonych danych

In [ ]:
df_final.to_csv(OUTPUT_DIR / "dane_obrobione.csv", index=False, sep=";")


## 4. Eksploracyjna analiza danych (EDA)

In [ ]:
braki_kolumn = df_final.isna().mean().sort_values(ascending=False) * 100

plt.figure(figsize=(8, 5))
braki_kolumn.plot(kind="barh", color="indianred")
plt.xlabel("% braków")
plt.title("Braki danych wg kolumny (cały zbiór)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
kategorie = sorted(df_final['kategoria'].unique())
fig, axes = plt.subplots(1, len(kategorie), figsize=(18, 5))

for ax, kat in zip(axes, kategorie):
    sub = df_clean[df_clean['kategoria'] == kat]['cena_brutto']
    limit = sub.quantile(0.99)
    sub_plot = sub[sub <= limit]
    ax.hist(sub_plot, bins=50, color='#4C72B0', edgecolor='white')
    ax.set_title(f"{kat}\n(n={len(sub)})")
    ax.set_xlabel("cena_brutto [zł]")
    ax.set_ylabel("liczba transakcji")
    ax.ticklabel_format(style='plain', axis='both')   # <-- obie osie na raz

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "histogramy_cena_brutto.png", dpi=150)

In [ ]:
progi = [0, 100_000, 300_000, 500_000, 1_000_000, float("inf")]
etykiety = ["<100k", "100k-300k", "300k-500k", "500k-1M", ">1M"]

df_final["segment_cenowy"] = pd.cut(df_final["cena_brutto"], bins=progi, labels=etykiety)

segmenty = df_final["segment_cenowy"].value_counts(normalize=True).sort_index() * 100

plt.figure(figsize=(8,5))
segmenty.plot(kind="bar", color="mediumpurple")
plt.ylabel("% transakcji")
plt.title("Udział transakcji wg segmentu cenowego")
plt.tight_layout()
plt.show()

In [ ]:
roczna_cena = (
    df_final
    .groupby(["rok", "kategoria"])["cena_brutto"]
    .median()
    .unstack("kategoria")
)

plt.figure(figsize=(10, 5))

for kategoria in roczna_cena.columns:
    plt.plot(
        roczna_cena.index,
        roczna_cena[kategoria],
        marker="o",
        linewidth=2,
        label=kategoria
    )

plt.xlabel("Rok")
plt.ylabel("Mediana ceny brutto")
plt.title("Mediana ceny brutto według kategorii nieruchomości")
plt.legend(title="Kategoria")
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# segmenty cenowe
progi = [0, 100_000, 300_000, 500_000, 1_000_000, float("inf")]
etykiety = ["<100k", "100k-300k", "300k-500k", "500k-1M", ">1M"]

df_final["segment_cenowy"] = pd.cut(
    df_final["cena_brutto"],
    bins=progi,
    labels=etykiety,
    include_lowest=True
)

# tabela: wiersze = rodzaj nieruchomości,
# kolumny = segment cenowy, wartości = liczba transakcji
udzial = (
    df_final
    .groupby(
        ["rodzaj_nieruchomosci", "segment_cenowy"],
        observed=True
    )
    .size()
    .unstack("segment_cenowy")
)

# procenty
udzial_proc = udzial.div(udzial.sum(axis=1), axis=0) * 100

# Uzupełnij brakujące segmenty zerami i zachowaj kolejność
udzial_proc = udzial_proc.reindex(columns=etykiety, fill_value=0)

# --- wykres ---
kolory = [
    '#b5ffb9',
    '#f9bc86',
    '#a3acff',
    '#ff9999',
    '#c9a0dc'
]

r = range(len(udzial_proc))
names = udzial_proc.index.tolist()
barWidth = 0.85

plt.figure(figsize=(10, 6))
dolne = np.zeros(len(udzial_proc))

for kolor, segment in zip(kolory, etykiety):
    wartosci = udzial_proc[segment].values

    plt.bar(
        r,
        wartosci,
        bottom=dolne,
        color=kolor,
        edgecolor='white',
        width=barWidth,
        label=segment
    )

    dolne += wartosci

plt.xticks(r, names, rotation=30, ha="right")
plt.xlabel("Rodzaj nieruchomości")
plt.ylabel("% transakcji")
plt.title("Udział segmentów cenowych wg rodzaju nieruchomości")

plt.legend(
    title="Segment cenowy",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

## 5. Przygotowanie zbioru danych do dalszych analiz

In [ ]:
df = df_final[
    (df_final['cena_brutto'] >= 100) &
    (df_final['cena_brutto'] < 1_300_000) &
    (df_final['powierzchnia'] >= 1)
].copy()

print(f"Liczba rekordów w zbiorze do analiz: {len(df)}")
print("  lokale: ", (df['kategoria'] == 'transakcje_lokale').sum())
print("  budynki:", (df['kategoria'] == 'transakcje_budynki').sum())
print("  działki:", (df['kategoria'] == 'transakcje_dzialki').sum())


## 6. Analiza wzrostu cen wg źródła danych (stabilne lokalizacje)

In [ ]:
# Zbiór roboczy do dalszych analiz
df = df_final

# 2. Formatowanie daty i wyciągnięcie roku
df['data_transakcji'] = pd.to_datetime(df['data_transakcji'])
df['rok'] = df['data_transakcji'].dt.year

# 4. Filtr stabilności: min. 30 transakcji W KAŻDYM ROKU dla danego źródła
transakcje_roczne = df.groupby(['zrodlo_plik', 'rok']).size().unstack(fill_value=0)
stabilne_zrodla = transakcje_roczne[(transakcje_roczne >= 1000).all(axis=1)].index

# 5. Wyliczenie median ceny_brutto tylko dla stabilnych plików
df_filtered = df[df['zrodlo_plik'].isin(stabilne_zrodla)]
mediany = df_filtered.groupby(['zrodlo_plik', 'rok'])['cena_brutto'].median().unstack()

# 6. Wyliczenie wzrostu % (ostatni rok vs pierwszy rok w danych)
min_rok = mediany.columns.min()
max_rok = mediany.columns.max()

wzrosty_realne = ((mediany[max_rok] - mediany[min_rok]) / mediany[min_rok]) * 100

print(f"REALNE TOP 10 (Wzrost % z {min_rok} do {max_rok} roku):")
print(wzrosty_realne.sort_values(ascending=False).head(10))

## 7. Wzbogacenie danych — dołączenie kolumny `dzi_sposob_uzyt` z pliku `tr2.csv`

In [ ]:
# Wczytanie dodatkowego pliku
tr2 = pd.read_csv(DATA_DIR / "tr2.csv")

# Sprawdzenie duplikatów id_transakcji w tr2
print("Liczba wierszy w tr2:", len(tr2))
print("Liczba unikalnych id_transakcji w tr2:", tr2["id_transakcji"].nunique())
print("Duplikaty id_transakcji:", tr2["id_transakcji"].duplicated().sum())

# Deduplikacja - zostawiamy pierwsze wystąpienie każdego id_transakcji
tr2_slim = tr2[["id_transakcji", "dzi_sposob_uzyt"]].drop_duplicates(subset="id_transakcji", keep="first")

# Jeśli kolumna już istnieje w df (np. z wcześniejszego, nieudanego merge'a) - usuń ją najpierw
if "dzi_sposob_uzyt" in df.columns:
    df = df.drop(columns=["dzi_sposob_uzyt"])

# Merge po id_transakcji (left join - zachowuje wszystkie wiersze z df)
df = df.merge(tr2_slim, on="id_transakcji", how="left")

# Przesunięcie nowej kolumny zaraz za kolumną "id_transakcji"
cols = list(df.columns)
cols.remove("dzi_sposob_uzyt")
idx = cols.index("id_transakcji") + 1
cols = cols[:idx] + ["dzi_sposob_uzyt"] + cols[idx:]
df = df[cols]

print("Liczba wierszy po merge:", len(df))
df.head()

In [ ]:
df.to_csv(OUTPUT_DIR / "dane_obrobione.csv", index=False, sep=";")


In [ ]:
# Sprawdzenie ile wartości faktycznie się wypełniło po merge
print("Braki w dzi_sposob_uzyt:", df['dzi_sposob_uzyt'].isna().sum())
print("Wypełnione:", df['dzi_sposob_uzyt'].notna().sum())
print(df['dzi_sposob_uzyt'].value_counts(dropna=False).head(10))

In [ ]:
# Sprawdzenie aktualnej kolejności kolumn
print(df.columns.tolist())

# Przesunięcie dzi_sposob_uzyt na 3. miejsce (index 2)
cols = list(df.columns)
cols.remove("dzi_sposob_uzyt")
cols.insert(2, "dzi_sposob_uzyt")
df = df[cols]

print("Nowa kolejność:", df.columns.tolist())

In [ ]:
print("Przed filtrem:", len(df))

df = df[df['dzi_sposob_uzyt'].notna()].copy()

print("Po filtrze (tylko wiersze z wartością w dzi_sposob_uzyt):", len(df))
df.head()

In [ ]:
df.to_csv(OUTPUT_DIR / "dane_obrobione.csv", index=False, sep=";")


### Dołączenie kolumny `dzi_przezn_wmpzp` (przeznaczenie w MPZP)

In [ ]:
# Dołączenie kolumny dzi_przezn_wmpzp (przeznaczenie w MPZP) po id_transakcji
tr2_slim2 = tr2[["id_transakcji", "dzi_przezn_wmpzp"]].drop_duplicates(subset="id_transakcji", keep="first")

# Jeśli kolumna już istnieje w df (np. z wcześniejszego, nieudanego merge'a) - usuń ją najpierw
if "dzi_przezn_wmpzp" in df.columns:
    df = df.drop(columns=["dzi_przezn_wmpzp"])

# Merge po id_transakcji (left join - zachowuje wszystkie wiersze z df)
# domyślnie merge dokleja nową kolumnę na koniec dataframe'u
df = df.merge(tr2_slim2, on="id_transakcji", how="left")

print("Liczba wierszy po merge:", len(df))
df.head()

In [ ]:
df.to_csv(OUTPUT_DIR / "dane_obrobione.csv", index=False, sep=";")


In [ ]:
# 50 najliczniejszych wartości w dzi_przezn_wmpzp
najczestsze_wartosci = df["dzi_przezn_wmpzp"].value_counts().head(50)
print(najczestsze_wartosci)

### Rozbicie wielowartościowych kombinacji na pojedyncze tokeny

In [ ]:
# Rozbicie wszystkich kombinacji po ";" na pojedyncze, unikalne wartości (tokeny)
wszystkie_tokeny = (
    df["dzi_przezn_wmpzp"]
    .dropna()
    .str.split(";")
    .explode()
    .str.strip()
)

# Zliczenie wystąpień każdego pojedynczego tokenu (niezależnie w ilu kombinacjach się pojawił)
tokeny_liczebnosc = wszystkie_tokeny.value_counts()

print(f"Liczba unikalnych wartości (tokenów): {tokeny_liczebnosc.nunique()}")
print(tokeny_liczebnosc)

In [ ]:
# Rozbicie kombinacji na osobne wiersze (jeden token = jeden wiersz)
df_wmpzp_long = df.copy()
df_wmpzp_long["dzi_przezn_wmpzp"] = df_wmpzp_long["dzi_przezn_wmpzp"].str.split(";")
df_wmpzp_long = df_wmpzp_long.explode("dzi_przezn_wmpzp")
df_wmpzp_long["dzi_przezn_wmpzp"] = df_wmpzp_long["dzi_przezn_wmpzp"].str.strip()

print("Liczba wierszy przed:", len(df))
print("Liczba wierszy po explode:", len(df_wmpzp_long))

In [ ]:
# Sprawdzenie ile braków w dzi_przezn_wmpzp przed usunięciem
print("Braki w dzi_przezn_wmpzp:", df_wmpzp_long["dzi_przezn_wmpzp"].isna().sum())
print("Wiersze przed usunięciem:", len(df_wmpzp_long))

df_wmpzp_long = df_wmpzp_long.dropna(subset=["dzi_przezn_wmpzp"])

print("Wiersze po usunięciu:", len(df_wmpzp_long))


In [ ]:
df_wmpzp_long.to_csv(OUTPUT_DIR / "dane_wmpzp_long.csv", index=False, sep=";")


## 8. Mediana cen wg sposobu użytkowania w czasie (tabele kwartalne)

In [ ]:
df_su = df.copy()
df_su["kwartal"] = df_su["data_transakcji"].dt.to_period("Q")

# Mediana cena_brutto wg sposobu użytkowania i kwartału
mediana_wide = (
    df_su.groupby(["dzi_sposob_uzyt", "kwartal"])["cena_brutto"]
    .median()
    .unstack("kwartal")
    .sort_index(axis=1)
)

liczba_wide = (
    df_su.groupby(["dzi_sposob_uzyt", "kwartal"])["cena_brutto"]
    .size()
    .unstack("kwartal")
    .sort_index(axis=1)
)

# Zmiana r/r = porównanie z tym samym kwartałem rok wcześniej
def rok_do_roku(wide_df):
    rr = pd.DataFrame(index=wide_df.index, columns=wide_df.columns, dtype=float)
    for col in wide_df.columns:
        poprzedni = pd.Period(year=col.year - 1, quarter=col.quarter, freq="Q")
        if poprzedni in wide_df.columns:
            rr[col] = (wide_df[col] - wide_df[poprzedni]) / wide_df[poprzedni] * 100
    return rr

mediana_rr = rok_do_roku(mediana_wide)

MIN_N = 10  # minimalna liczba transakcji, żeby mediana miała sens

# Kwartały od 2022 (z pominięciem niepełnego 2026Q4)
kwartaly_2024plus = sorted(
    k for k in mediana_wide.columns
    if k.year >= 2022 and k != pd.Period("2026Q4", freq="Q")
)


In [ ]:
def zbuduj_tabele(kwartal, min_n=MIN_N):
    tabela = pd.DataFrame({
        "mediana": mediana_wide.get(kwartal),
        "r/r": mediana_rr.get(kwartal),
        "n": liczba_wide.get(kwartal),
    })
    tabela = tabela[tabela["n"] >= min_n]
    tabela = tabela.sort_values("mediana", ascending=False).reset_index()
    tabela.columns = ["Sposób użytkowania", "Mediana", "r/r", "Liczba transakcji"]
    return tabela

def formatuj_rr(val):
    if pd.isna(val):
        return "—"
    strzalka = "▲" if val >= 0 else "▼"
    return f"{strzalka} {val:+.1f}%"

def koloruj_rr(val):
    if pd.isna(val):
        return ""
    kolor_tla = "#d4f4dd" if val >= 0 else "#fbdada"
    kolor_tekstu = "#1a7a3c" if val >= 0 else "#c0392b"
    return (
        f"background-color: {kolor_tla}; color: {kolor_tekstu}; "
        "font-weight: 600; border-radius: 999px; padding: 6px 12px; "
        "text-align: center;"
    )

def styluj(tabela, tytul=None):
    styl = (
        tabela.style
        .hide(axis="index")
        .format({"Mediana": "{:,.0f} zł", "r/r": formatuj_rr, "Liczba transakcji": "{:,.0f}"}, na_rep="—")
        .map(koloruj_rr, subset=["r/r"])
        .set_properties(subset=["Sposób użytkowania"], **{"font-weight": "700"})
        .set_properties(subset=["Mediana"], **{"font-weight": "700"})
        .set_table_styles([
            {"selector": "th", "props": [
                ("background-color", "#1f2937"), ("color", "white"),
                ("padding", "10px 16px"), ("text-align", "left"),
                ("font-family", "Segoe UI, Arial, sans-serif"), ("font-size", "13px")]},
            {"selector": "td", "props": [
                ("padding", "10px 16px"), ("font-family", "Segoe UI, Arial, sans-serif"),
                ("font-size", "14px"), ("border-bottom", "1px solid #eee")]},
            {"selector": "tr:hover", "props": [("background-color", "#f9fafb")]},
            {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%")]},
        ])
    )
    if tytul:
        styl = styl.set_caption(tytul).set_table_styles(
            [{"selector": "caption", "props": [
                ("caption-side", "top"), ("font-size", "17px"), ("font-weight", "700"),
                ("padding", "4px 0 12px 0"), ("text-align", "left"),
                ("font-family", "Segoe UI, Arial, sans-serif")]}],
            overwrite=False
        )
    return styl

for kw in kwartaly_2024plus:
    display(styluj(zbuduj_tabele(kw), tytul=f"Sposób użytkowania — {kw}"))

## 9. Generowanie interaktywnego raportu HTML

Raport (`raport_interaktywny.html`) budowany jest w dwóch krokach:
1. tabele mediany/średniej cen wg sposobu użytkowania x przeznaczenia w MPZP (kwartalnie i miesięcznie),
2. dołączenie sekcji z medianą cen wg rejonu (dopisywanej do istniejącego pliku).

In [ ]:
# ============================================================
# 1) Dane połączone: sposób użytkowania x przeznaczenie x KWARTAŁ
# ============================================================
df_comb = df.dropna(subset=["dzi_przezn_wmpzp", "dzi_sposob_uzyt"]).copy()
df_comb["kwartal"] = df_comb["data_transakcji"].dt.to_period("Q")
df_comb["dzi_przezn_wmpzp"] = df_comb["dzi_przezn_wmpzp"].str.split(";")
df_comb = df_comb.explode("dzi_przezn_wmpzp")
df_comb["dzi_przezn_wmpzp"] = df_comb["dzi_przezn_wmpzp"].str.strip()

mediana_comb = (
    df_comb.groupby(["dzi_sposob_uzyt", "dzi_przezn_wmpzp", "kwartal"])["cena_brutto"]
    .median().unstack("kwartal").sort_index(axis=1)
)
srednia_comb = (
    df_comb.groupby(["dzi_sposob_uzyt", "dzi_przezn_wmpzp", "kwartal"])["cena_brutto"]
    .mean().unstack("kwartal").sort_index(axis=1)
)
liczba_comb = (
    df_comb.groupby(["dzi_sposob_uzyt", "dzi_przezn_wmpzp", "kwartal"])["cena_brutto"]
    .size().unstack("kwartal").sort_index(axis=1)
)
rr_mediana_comb = rok_do_roku(mediana_comb)
rr_srednia_comb = rok_do_roku(srednia_comb)

dzisiaj = pd.Timestamp.now()
biezacy_kwartal = pd.Period(dzisiaj, freq="Q")
kwartaly_comb = sorted(k for k in mediana_comb.columns if k.year >= 2023 and k < biezacy_kwartal)

MIN_N = 10

rekordy = []
for kw in kwartaly_comb:
    for (sposob, przezn) in mediana_comb.index:
        n = liczba_comb.loc[(sposob, przezn), kw] if kw in liczba_comb.columns else None
        if pd.isna(n) or n < MIN_N:
            continue
        rekordy.append({
            "sposob": sposob,
            "przeznaczenie": przezn,
            "kwartal": str(kw),
            "srednia": None if pd.isna(srednia_comb.loc[(sposob, przezn), kw]) else round(srednia_comb.loc[(sposob, przezn), kw]),
            "srednia_rr": None if pd.isna(rr_srednia_comb.loc[(sposob, przezn), kw]) else round(rr_srednia_comb.loc[(sposob, przezn), kw], 1),
            "mediana": None if pd.isna(mediana_comb.loc[(sposob, przezn), kw]) else round(mediana_comb.loc[(sposob, przezn), kw]),
            "mediana_rr": None if pd.isna(rr_mediana_comb.loc[(sposob, przezn), kw]) else round(rr_mediana_comb.loc[(sposob, przezn), kw], 1),
            "n": int(n),
        })

sposoby_lista = sorted({r["sposob"] for r in rekordy})
przeznaczenia_lista = sorted({r["przeznaczenie"] for r in rekordy})
kwartaly_lista = [str(k) for k in kwartaly_comb]
dane_json = json.dumps(rekordy, ensure_ascii=False)

# ============================================================
# 2) Dane połączone: sposób użytkowania x przeznaczenie x MIESIĄC (od 2023)
# ============================================================
df_comb_m = df.dropna(subset=["dzi_przezn_wmpzp", "dzi_sposob_uzyt"]).copy()
df_comb_m["miesiac"] = df_comb_m["data_transakcji"].dt.to_period("M")
df_comb_m["dzi_przezn_wmpzp"] = df_comb_m["dzi_przezn_wmpzp"].str.split(";")
df_comb_m = df_comb_m.explode("dzi_przezn_wmpzp")
df_comb_m["dzi_przezn_wmpzp"] = df_comb_m["dzi_przezn_wmpzp"].str.strip()

mediana_comb_m = (
    df_comb_m.groupby(["dzi_sposob_uzyt", "dzi_przezn_wmpzp", "miesiac"])["cena_brutto"]
    .median().unstack("miesiac").sort_index(axis=1)
)
srednia_comb_m = (
    df_comb_m.groupby(["dzi_sposob_uzyt", "dzi_przezn_wmpzp", "miesiac"])["cena_brutto"]
    .mean().unstack("miesiac").sort_index(axis=1)
)
liczba_comb_m = (
    df_comb_m.groupby(["dzi_sposob_uzyt", "dzi_przezn_wmpzp", "miesiac"])["cena_brutto"]
    .size().unstack("miesiac").sort_index(axis=1)
)

def rok_do_roku_miesiac(wide_df):
    rr = pd.DataFrame(index=wide_df.index, columns=wide_df.columns, dtype=float)
    for col in wide_df.columns:
        poprzedni = col - 12
        if poprzedni in wide_df.columns:
            rr[col] = (wide_df[col] - wide_df[poprzedni]) / wide_df[poprzedni] * 100
    return rr

rr_mediana_comb_m = rok_do_roku_miesiac(mediana_comb_m)
rr_srednia_comb_m = rok_do_roku_miesiac(srednia_comb_m)

biezacy_miesiac = pd.Period(dzisiaj, freq="M")
miesiace_comb = sorted(k for k in mediana_comb_m.columns if k.year >= 2023 and k < biezacy_miesiac)

MIN_N_MIESIAC = 5

rekordy_m = []
for mc in miesiace_comb:
    for (sposob, przezn) in mediana_comb_m.index:
        n = liczba_comb_m.loc[(sposob, przezn), mc] if mc in liczba_comb_m.columns else None
        if pd.isna(n) or n < MIN_N_MIESIAC:
            continue
        rekordy_m.append({
            "sposob": sposob,
            "przeznaczenie": przezn,
            "miesiac": str(mc),
            "srednia": None if pd.isna(srednia_comb_m.loc[(sposob, przezn), mc]) else round(srednia_comb_m.loc[(sposob, przezn), mc]),
            "srednia_rr": None if pd.isna(rr_srednia_comb_m.loc[(sposob, przezn), mc]) else round(rr_srednia_comb_m.loc[(sposob, przezn), mc], 1),
            "mediana": None if pd.isna(mediana_comb_m.loc[(sposob, przezn), mc]) else round(mediana_comb_m.loc[(sposob, przezn), mc]),
            "mediana_rr": None if pd.isna(rr_mediana_comb_m.loc[(sposob, przezn), mc]) else round(rr_mediana_comb_m.loc[(sposob, przezn), mc], 1),
            "n": int(n),
        })

sposoby_m_lista = sorted({r["sposob"] for r in rekordy_m})
przeznaczenia_m_lista = sorted({r["przeznaczenie"] for r in rekordy_m})
miesiace_lista = [str(k) for k in miesiace_comb]
dane_m_json = json.dumps(rekordy_m, ensure_ascii=False)

# ============================================================
# 3) Sklejanie HTML — dwie tabele (kwartalna + miesięczna), Średnia + Mediana
# ============================================================
html = f"""<!DOCTYPE html>
<html lang="pl"><head><meta charset="utf-8">
<style>
  body {{ font-family: 'Segoe UI', Arial, sans-serif; background:#f4f5f7; padding:30px; }}
  .panel {{ background:white; border-radius:12px; padding:20px; margin-bottom:20px; box-shadow:0 1px 3px rgba(0,0,0,.08); }}
  select {{ padding:6px; border-radius:6px; border:1px solid #ccc; min-width:220px; }}
  table {{ width:100%; border-collapse:collapse; margin-top:16px; }}
  th {{ background:#1f2937; color:white; text-align:left; padding:10px 16px; font-size:13px; }}
  td {{ padding:10px 16px; border-bottom:1px solid #eee; font-size:14px; }}
  tr:hover td {{ background:#f9fafb; }}
  .pill {{ padding:4px 12px; border-radius:999px; font-weight:600; display:inline-block; }}
  .up {{ background:#d4f4dd; color:#1a7a3c; }}
  .down {{ background:#fbdada; color:#c0392b; }}
  .checks label {{ margin-right:14px; font-size:14px; }}
  .filtry {{ display:flex; gap:24px; flex-wrap:wrap; align-items:flex-start; }}
  .filtr-blok {{ min-width:220px; }}
  .filtr-blok > span {{ display:block; font-weight:600; margin-bottom:6px; font-size:13px; color:#374151; }}
  hr {{ border:none; border-top:1px solid #ddd; margin:36px 0; }}
</style></head>
<body>
  <h2>Mediana i średnia cen — sposób użytkowania x przeznaczenie działki (kwartalnie, od 2023)</h2>
  <div class="panel">
    <div class="filtry">
      <div class="filtr-blok"><span>Kwartał</span><select id="kwartal"></select></div>
      <div class="filtr-blok"><span>Przeznaczenie</span><select id="przeznaczenie"></select></div>
      <div class="filtr-blok" style="flex:1;">
        <span>Sposób użytkowania</span>
        <div class="checks" id="checks"></div>
      </div>
    </div>
  </div>
  <div class="panel"><table id="tabela"><thead><tr>
    <th>Sposób użytkowania</th><th>Średnia</th><th>r/r</th><th>Mediana</th><th>r/r</th><th>Liczba transakcji</th>
  </tr></thead><tbody id="tbody"></tbody></table></div>

  <hr>

  <h2>Mediana i średnia cen — sposób użytkowania x przeznaczenie działki (miesięcznie, od 2023)</h2>
  <div class="panel">
    <div class="filtry">
      <div class="filtr-blok"><span>Miesiąc</span><select id="miesiac"></select></div>
      <div class="filtr-blok"><span>Przeznaczenie</span><select id="przeznaczenieM"></select></div>
      <div class="filtr-blok" style="flex:1;">
        <span>Sposób użytkowania</span>
        <div class="checks" id="checksM"></div>
      </div>
    </div>
  </div>
  <div class="panel"><table id="tabelaM"><thead><tr>
    <th>Sposób użytkowania</th><th>Średnia</th><th>r/r</th><th>Mediana</th><th>r/r</th><th>Liczba transakcji</th>
  </tr></thead><tbody id="tbodyM"></tbody></table></div>

<script>
function pillHtml(rr) {{
  if (rr === null) return "—";
  return `<span class="pill ${{rr>=0?'up':'down'}}">${{rr>=0?'▲':'▼'}} ${{rr>0?'+':''}}${{rr}}%</span>`;
}}
function kwotaHtml(v) {{
  return v === null ? "—" : `${{v.toLocaleString('pl-PL')}} zł`;
}}

// --- Sekcja kwartalna ---
const dane = {dane_json};
const sposoby = {json.dumps(sposoby_lista, ensure_ascii=False)};
const przeznaczenia = {json.dumps(przeznaczenia_lista, ensure_ascii=False)};
const kwartaly = {json.dumps(kwartaly_lista)};

const selKwartal = document.getElementById("kwartal");
kwartaly.forEach(k => selKwartal.innerHTML += `<option value="${{k}}">${{k}}</option>`);
selKwartal.value = kwartaly[kwartaly.length - 1];

const selPrzeznaczenie = document.getElementById("przeznaczenie");
przeznaczenia.forEach(p => selPrzeznaczenie.innerHTML += `<option value="${{p}}">${{p}}</option>`);

const checks = document.getElementById("checks");
sposoby.forEach(s => {{
  checks.innerHTML += `<label><input type="checkbox" value="${{s}}" checked> ${{s}}</label>`;
}});

function rysuj() {{
  const kw = selKwartal.value;
  const przezn = selPrzeznaczenie.value;
  const wybraneSposoby = [...checks.querySelectorAll("input:checked")].map(c => c.value);

  const wiersze = dane.filter(r =>
    r.kwartal === kw && r.przeznaczenie === przezn && wybraneSposoby.includes(r.sposob)
  ).sort((a, b) => (b.srednia || 0) - (a.srednia || 0));

  document.getElementById("tbody").innerHTML = wiersze.map(r => `
    <tr><td><b>${{r.sposob}}</b></td>
    <td><b>${{kwotaHtml(r.srednia)}}</b></td><td>${{pillHtml(r.srednia_rr)}}</td>
    <td><b>${{kwotaHtml(r.mediana)}}</b></td><td>${{pillHtml(r.mediana_rr)}}</td>
    <td>${{r.n}}</td></tr>`).join("") ||
    `<tr><td colspan="6">Brak danych dla tej kombinacji (za mało transakcji — próg minimum {MIN_N})</td></tr>`;
}}

selKwartal.addEventListener("change", rysuj);
selPrzeznaczenie.addEventListener("change", rysuj);
checks.addEventListener("change", rysuj);
rysuj();

// --- Sekcja miesięczna ---
const daneM = {dane_m_json};
const sposobyM = {json.dumps(sposoby_m_lista, ensure_ascii=False)};
const przeznaczeniaM = {json.dumps(przeznaczenia_m_lista, ensure_ascii=False)};
const miesiace = {json.dumps(miesiace_lista)};

const selMiesiac = document.getElementById("miesiac");
miesiace.forEach(m => selMiesiac.innerHTML += `<option value="${{m}}">${{m}}</option>`);
selMiesiac.value = miesiace[miesiace.length - 1];

const selPrzeznaczenieM = document.getElementById("przeznaczenieM");
przeznaczeniaM.forEach(p => selPrzeznaczenieM.innerHTML += `<option value="${{p}}">${{p}}</option>`);

const checksM = document.getElementById("checksM");
sposobyM.forEach(s => {{
  checksM.innerHTML += `<label><input type="checkbox" value="${{s}}" checked> ${{s}}</label>`;
}});

function rysujM() {{
  const mc = selMiesiac.value;
  const przezn = selPrzeznaczenieM.value;
  const wybraneSposoby = [...checksM.querySelectorAll("input:checked")].map(c => c.value);

  const wiersze = daneM.filter(r =>
    r.miesiac === mc && r.przeznaczenie === przezn && wybraneSposoby.includes(r.sposob)
  ).sort((a, b) => (b.srednia || 0) - (a.srednia || 0));

  document.getElementById("tbodyM").innerHTML = wiersze.map(r => `
    <tr><td><b>${{r.sposob}}</b></td>
    <td><b>${{kwotaHtml(r.srednia)}}</b></td><td>${{pillHtml(r.srednia_rr)}}</td>
    <td><b>${{kwotaHtml(r.mediana)}}</b></td><td>${{pillHtml(r.mediana_rr)}}</td>
    <td>${{r.n}}</td></tr>`).join("") ||
    `<tr><td colspan="6">Brak danych dla tej kombinacji (za mało transakcji — próg minimum {MIN_N_MIESIAC})</td></tr>`;
}}

selMiesiac.addEventListener("change", rysujM);
selPrzeznaczenieM.addEventListener("change", rysujM);
checksM.addEventListener("change", rysujM);
rysujM();
</script>
</body></html>"""

with open(OUTPUT_DIR / "raport_interaktywny.html", "w", encoding="utf-8") as f:
    f.write(html)

print("Zapisano: raport_interaktywny.html")

### Dołączenie sekcji regionalnej (mediana cen wg rejonu) do raportu

In [ ]:
# ============================================================
# 1) Dane: mediana cena_brutto per rejon x rok, dla WSZYSTKICH rejonów
# ============================================================
df_reg = df.copy()
df_reg["rejon"] = df_reg["zrodlo_plik"].str[:-5]
df_reg["rok"] = df_reg["data_transakcji"].dt.year

rok_biezacy = pd.Timestamp.now().year
df_reg = df_reg[(df_reg["rok"] >= 2021) & (df_reg["rok"] < rok_biezacy)]

mediana_rok = (
    df_reg.groupby(["rejon", "rok"])["cena_brutto"]
    .median()
    .unstack("rok")
    .sort_index(axis=1)
)
liczba_rok = (
    df_reg.groupby(["rejon", "rok"])["cena_brutto"]
    .size()
    .unstack("rok")
    .sort_index(axis=1)
)

lata_lista = mediana_rok.columns.tolist()
ostatni_rok = lata_lista[-1]

MIN_N_ROK = 10  # minimalna liczba transakcji w roku, żeby punkt się liczył

dane_rejony = {}
for rejon in mediana_rok.index:
    punkty = []
    for rok in lata_lista:
        n = liczba_rok.loc[rejon, rok] if rok in liczba_rok.columns else 0
        wartosc = mediana_rok.loc[rejon, rok]
        if pd.isna(n) or n < MIN_N_ROK or pd.isna(wartosc):
            continue
        punkty.append({"rok": int(rok), "mediana": round(wartosc)})
    if len(punkty) >= 2:
        dane_rejony[rejon] = punkty

rejony_lista = sorted(dane_rejony.keys())
dane_rejony_json = json.dumps(dane_rejony, ensure_ascii=False)

# Domyślnie zaznaczone rejony — największe miasta PL, tylko jeśli są dostępne w danych
PREFEROWANE_DOMYSLNE = [
    "miasto_stoleczne_warszawa", "miasto_krakow", "miasto_lodz",
    "miasto_wroclaw", "miasto_poznan", "miasto_gdansk",
]
domyslne_rejony = [r for r in PREFEROWANE_DOMYSLNE if r in dane_rejony][:6]
if len(domyslne_rejony) < 6:
    for r in rejony_lista:
        if r not in domyslne_rejony:
            domyslne_rejony.append(r)
        if len(domyslne_rejony) == 6:
            break

domyslne_rejony_json = json.dumps(domyslne_rejony, ensure_ascii=False)

# ============================================================
# 2) Sekcja HTML/JS: wyszukiwarka + lista z checkboxami (max 6)
#    + panel wybranych (chipy z x) + karty z SVG sparkline
# ============================================================
sekcja_karty = f"""
  <hr>
  <style>
    .rejon-panel {{ display:flex; gap:24px; flex-wrap:wrap; margin-bottom:20px; }}
    .rejon-szukaj-blok {{ min-width:280px; flex:1; }}
    .rejon-szukaj-blok input[type="text"] {{
      width:100%; padding:8px 12px; border:1px solid #ccc; border-radius:8px; font-size:14px; box-sizing:border-box;
    }}
    .rejon-licznik {{ font-size:12px; color:#6b7280; margin:6px 0; }}
    .rejon-lista {{
      max-height:220px; overflow-y:auto; border:1px solid #e5e7eb; border-radius:8px;
      padding:8px 12px; background:white; margin-top:6px;
    }}
    .rejon-lista label {{ display:block; padding:4px 0; font-size:14px; }}
    .rejon-lista label.wylaczony {{ color:#c7cbd1; }}
    .rejon-wybrane-blok {{ min-width:220px; flex:1; }}
    .rejon-wybrane-naglowek {{ font-size:12px; color:#6b7280; margin-bottom:6px; }}
    .rejon-wybrane-lista {{ display:flex; flex-wrap:wrap; gap:8px; align-content:flex-start; }}
    .chip {{
      display:flex; align-items:center; gap:6px; background:#ede9fe; color:#2d1b69;
      border-radius:999px; padding:6px 8px 6px 12px; font-size:13px; font-weight:600;
    }}
    .chip-usun {{
      cursor:pointer; background:none; border:none; color:#2d1b69; font-size:15px;
      line-height:1; padding:0 2px; font-weight:700;
    }}
    .chip-usun:hover {{ color:#c1121f; }}
    .rejon-brak-wybranych {{ color:#9ca3af; font-size:13px; }}
    .siatka {{ display:grid; grid-template-columns: repeat(3, 1fr); gap:16px; max-width:1200px; }}
    .karta {{ background:white; border:1px solid #e5e7eb; border-radius:14px; padding:18px 20px; }}
    .karta-naglowek {{ display:flex; justify-content:space-between; align-items:flex-start; margin-bottom:6px; }}
    .karta-tytul {{ font-weight:700; font-size:16px; color:#111827; }}
    .karta-wartosc-blok {{ text-align:right; }}
    .karta-wartosc {{ font-weight:700; font-size:15px; color:#111827; display:block; }}
    .karta-etykieta {{ font-size:11px; color:#9ca3af; }}
    .karta-wykres {{ width:100%; height:80px; display:block; margin-top:4px; }}
    .karta-pusta {{ color:#9ca3af; font-size:14px; padding:20px; }}
    @media (max-width: 900px) {{ .siatka {{ grid-template-columns: repeat(2, 1fr); }} }}
    @media (max-width: 600px) {{ .siatka {{ grid-template-columns: 1fr; }} }}
  </style>

  <h2>Mediana cen wg rejonu</h2>
  <div class="panel">
    <div class="rejon-panel">
      <div class="rejon-szukaj-blok">
        <input type="text" id="rejonSzukaj" placeholder="Szukaj rejonu...">
        <div class="rejon-licznik" id="rejonLicznik">Wybrano: 0 / 6</div>
        <div class="rejon-lista" id="rejonLista"></div>
      </div>
      <div class="rejon-wybrane-blok">
        <div class="rejon-wybrane-naglowek">Wybrane rejony (kliknij ×, żeby odznaczyć):</div>
        <div class="rejon-wybrane-lista" id="rejonWybraneLista"></div>
      </div>
    </div>
  </div>

  <div class="siatka" id="siatkaRejony"></div>
"""

skrypt_karty = f"""
const daneRejony = {dane_rejony_json};
const rejonyLista = {json.dumps(rejony_lista, ensure_ascii=False)};
const MAX_REJONOW = 6;
let wybraneRejony = {domyslne_rejony_json}.filter(r => rejonyLista.includes(r));

const inputSzukaj = document.getElementById("rejonSzukaj");
const listaDiv = document.getElementById("rejonLista");
const licznikDiv = document.getElementById("rejonLicznik");
const wybraneListaDiv = document.getElementById("rejonWybraneLista");
const siatkaDiv = document.getElementById("siatkaRejony");

function renderLista() {{
  const fraza = inputSzukaj.value.trim().toLowerCase();
  const pasujace = rejonyLista.filter(r => r.toLowerCase().includes(fraza));

  listaDiv.innerHTML = pasujace.map(r => {{
    const zaznaczony = wybraneRejony.includes(r);
    const zablokowany = !zaznaczony && wybraneRejony.length >= MAX_REJONOW;
    return `<label class="${{zablokowany ? 'wylaczony' : ''}}">
      <input type="checkbox" value="${{r}}" ${{zaznaczony ? 'checked' : ''}} ${{zablokowany ? 'disabled' : ''}}>
      ${{r}}
    </label>`;
  }}).join("") || `<div style="color:#9ca3af; font-size:13px;">Brak wyników</div>`;

  licznikDiv.textContent = `Wybrano: ${{wybraneRejony.length}} / ${{MAX_REJONOW}}`;
}}

function renderWybrane() {{
  if (wybraneRejony.length === 0) {{
    wybraneListaDiv.innerHTML = `<div class="rejon-brak-wybranych">Brak wybranych rejonów.</div>`;
    return;
  }}
  wybraneListaDiv.innerHTML = wybraneRejony.map(r => {{
    return `<span class="chip">${{r}}<button type="button" class="chip-usun" data-rejon="${{r}}" title="Odznacz">×</button></span>`;
  }}).join("");
}}

function odznacz(rejon) {{
  wybraneRejony = wybraneRejony.filter(r => r !== rejon);
  renderLista();
  renderWybrane();
  renderSiatka();
}}

listaDiv.addEventListener("change", (e) => {{
  if (e.target.tagName !== "INPUT") return;
  const val = e.target.value;
  if (e.target.checked) {{
    if (wybraneRejony.length < MAX_REJONOW) wybraneRejony.push(val);
  }} else {{
    wybraneRejony = wybraneRejony.filter(r => r !== val);
  }}
  renderLista();
  renderWybrane();
  renderSiatka();
}});

wybraneListaDiv.addEventListener("click", (e) => {{
  const btn = e.target.closest(".chip-usun");
  if (!btn) return;
  odznacz(btn.dataset.rejon);
}});

inputSzukaj.addEventListener("input", renderLista);

function rysujSvgSparkline(punkty) {{
  const w = 300, h = 70, padX = 6, padY = 8;
  const wartosci = punkty.map(p => p.mediana);
  const minV = Math.min(...wartosci), maxV = Math.max(...wartosci);
  const zakres = (maxV - minV) || 1;

  const krok = (w - padX * 2) / (punkty.length - 1);
  const pts = punkty.map((p, i) => {{
    const x = padX + i * krok;
    const y = h - padY - ((p.mediana - minV) / zakres) * (h - padY * 2);
    return [x, y];
  }});

  const linia = pts.map(([x, y]) => `${{x.toFixed(1)}},${{y.toFixed(1)}}`).join(" ");
  const etykiety = punkty.map((p, i) => {{
    const x = padX + i * krok;
    return `<text x="${{x}}" y="${{h - 2}}" font-size="8" fill="#9ca3af" text-anchor="middle">${{p.rok}}</text>`;
  }}).join("");

  return `<svg viewBox="0 0 ${{w}} ${{h}}" xmlns="http://www.w3.org/2000/svg">
    <polyline points="${{linia}}" fill="none" stroke="#2d1b69" stroke-width="2.4" stroke-linecap="round" stroke-linejoin="round"/>
    <line x1="${{padX}}" y1="${{h - 14}}" x2="${{w - padX}}" y2="${{h - 14}}" stroke="#e5e7eb" stroke-width="1"/>
    ${{etykiety}}
  </svg>`;
}}

function renderSiatka() {{
  if (wybraneRejony.length === 0) {{
    siatkaDiv.innerHTML = `<div class="karta-pusta">Wybierz maks. 6 rejonów z listy powyżej, żeby zobaczyć wykresy.</div>`;
    return;
  }}
  siatkaDiv.innerHTML = wybraneRejony.map(rejon => {{
    const punkty = daneRejony[rejon];
    const ostatni = punkty[punkty.length - 1];
    const svg = rysujSvgSparkline(punkty);
    return `<div class="karta">
      <div class="karta-naglowek">
        <span class="karta-tytul">${{rejon}}</span>
        <div class="karta-wartosc-blok">
          <span class="karta-wartosc">${{ostatni.mediana.toLocaleString('pl-PL')}} zł</span>
          <span class="karta-etykieta">mediana · ${{ostatni.rok}}</span>
        </div>
      </div>
      <div class="karta-wykres">${{svg}}</div>
    </div>`;
  }}).join("");
}}

renderLista();
renderWybrane();
renderSiatka();
"""

MARKER_START = '<!-- SEKCJA-REJONY-START -->'
MARKER_END = '<!-- SEKCJA-REJONY-END -->'

sekcja_pelna = MARKER_START + sekcja_karty + f"""
  <script>
  document.addEventListener("DOMContentLoaded", function() {{
    {skrypt_karty}
  }});
  </script>
""" + MARKER_END

# ============================================================
# 3) Wklejenie do istniejącego pliku raport_interaktywny.html
# ============================================================
with open(OUTPUT_DIR / "raport_interaktywny.html", "r", encoding="utf-8") as f:
    tresc = f.read()

# Usuń WSZYSTKIE poprzednie kopie sekcji "Mediana cen wg rejonu"
# — zarówno oznaczone markerami, jak i stare, sprzed poprawki
wzorzec = re.compile(
    r'(<!-- SEKCJA-REJONY-START -->.*?<!-- SEKCJA-REJONY-END -->'
    r'|<hr>\s*<style>.*?\.rejon-panel.*?</style>.*?<h2>Mediana cen wg rejonu(warunek conajmniej 10 transkacji)</h2>.*?</script>\s*)',
    re.DOTALL
)
tresc, ile_usunieto = wzorzec.subn("", tresc)
print(f"Usunięto {ile_usunieto} starych kopii sekcji")

assert tresc.count("</body></html>") == 1, "Więcej niż jedno wystąpienie </body></html> — sprawdź plik ręcznie"
tresc = tresc.replace("</body></html>", sekcja_pelna + "</body></html>")

with open(OUTPUT_DIR / "raport_interaktywny.html", "w", encoding="utf-8") as f:
    f.write(tresc)

print("Dopisano interaktywną sekcję rejonów do raport_interaktywny.html")
print("Liczba rejonów dostępnych do wyboru:", len(rejony_lista))
print("Domyślnie zaznaczone:", domyslne_rejony)

## Podsumowanie

Po wykonaniu wszystkich komórek w katalogu `output/` powinny znaleźć się:
- `dane_obrobione.csv` — finalny, oczyszczony i wzbogacony zbiór danych,
- `dane_wmpzp_long.csv` — dane w formacie długim wg przeznaczenia w MPZP,
- `histogramy_cena_brutto.png` — histogramy cen wg kategorii nieruchomości,
- `raport_interaktywny.html` — interaktywny raport HTML z tabelami i wykresami.